In [2]:
set_A = {'자연어','처리','머신','러닝','알고리즘'}
set_B = {'인공지능','머신','러닝','딥러닝','알고리즘'}
print(f'문서 A 크기 : {len(set_A)}')
print(f'문서 B 크기 : {len(set_B)}')

# 교집합 합집합
intersection = set_A.intersection(set_B)
union = set_A.union(set_B)

print(f'교집합 크기 : {len(intersection)}')
print(f'합집합 크기 : {len(union)}')

# 자카드 유사도
jacard_sim = len(intersection) / len(union)
# 다이스 유사도
dice_sim = (2*len(intersection)) / (len(set_A) / len(set_B))
print(f'자카드 유사도 : {jacard_sim} / 다이스 유사도 : {dice_sim}')
# 코사인 유사도
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
corpus = [
    '자연어 처리 머신 러닝 알고리즘',
    '인공지능 머신 러닝 딥러닝 알고리즘'
]
# 오타가있는 단어일경우 코사인유사도는 0
# corpus = [
#     'apple',
#     'aple'
# ]

tfidf = TfidfVectorizer()
tfidf_metrix = tfidf.fit_transform(corpus)
cos_sim = cosine_similarity(tfidf_metrix[0], tfidf_metrix[1])
print(f'코사인 유사도 : {cos_sim[0][0]:.3f}')

문서 A 크기 : 5
문서 B 크기 : 5
교집합 크기 : 3
합집합 크기 : 7
자카드 유사도 : 0.42857142857142855 / 다이스 유사도 : 6.0
코사인 유사도 : 0.432


In [4]:
text = 'monkey'; k = 2
# k+3 -1 <= len(text)

def get_shingles(text, k=2):
    '''단어를 n그램으로 분리하는 함수'''
    return {text[i : k + i] for i in range(len(text) - 1)}

word_a, word_b = 'apple', 'aple'
shingle_a, shingle_b = get_shingles(word_a), get_shingles(word_b)
print(f'{word_a}  {shingle_a}')
print(f'{word_b}  {shingle_b}')

inter = shingle_a.intersection(shingle_b)
uni = shingle_a.union(shingle_b)
sim = len(inter) / len(uni)
print(f'자카드 유사도 : {sim:.3f}')

apple  {'pp', 'ap', 'le', 'pl'}
aple  {'le', 'ap', 'pl'}
자카드 유사도 : 0.750


In [ ]:
# BOW 기반 문서 분류
news_data = {
    'content': [
        # IT/과학
        "삼성전자가 차세대 폴더블 스마트폰을 공개하며 시장 공략에 나섰습니다.",
        "인공지능 기술이 발전함에 따라 반도체 수요가 급증하고 있습니다.",
        "구글의 새로운 AI 모델이 인간과의 대화에서 자연스러운 반응을 보였습니다.",
        # 경제
        "미국 연준이 금리를 동결하면서 국내 증시는 혼조세를 보였습니다.",
        "최근 물가 상승률이 둔화되면서 하반기 경기 회복 기대감이 커지고 있습니다.",
        "대기업들의 실적 발표가 이어지는 가운데 반도체 업종이 강세를 보였습니다.",
        # 사회
        "이번 주말 서울 도심에서 대규모 집회가 예정되어 교통 혼잡이 예상됩니다.",
        "경찰은 최근 급증하는 보이스피싱 범죄를 막기 위해 집중 단속에 나섰습니다.",
        "정부는 저출산 문제 해결을 위해 새로운 육아 지원 정책을 발표했습니다.",
        # 정치
        "여야 정치권은 국회 본회의를 열고 민생 법안 처리에 합의했습니다.",
        "대통령은 국무회의에서 경제 활성화를 위한 규제 개혁을 강조했습니다.",
        "새로운 정당 창당 소식에 정치권의 판도가 요동치고 있습니다."
    ],
    'category': ['IT/과학', 'IT/과학', 'IT/과학', '경제', '경제', '경제', '사회', '사회', '사회', '정치', '정치', '정치']
}

In [6]:
from konlpy.tag import Okt
okt = Okt()

In [7]:
# tagging이 nouns이고 글자 길이가 2글자 이상인 단어들만 token화
[word for word in okt.nouns("삼성전자가 차세대 폴더블 스마트폰을 공개하며 시장 공략에 나섰습니다.") if len(word) <= 2]

['전자', '폴', '더블', '공개', '시장', '공략']

In [8]:
# 한글 토크나이저
def korean_tokenizer(text):
    okt = Okt()
    return [word for word in okt.nouns(text) if len(word) >= 2]

In [9]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
tfidf = TfidfVectorizer(tokenizer=korean_tokenizer)
x, y = news_data['content'], news_data['category']
x_tfidf = tfidf.fit_transform(x)

x_train, x_test, y_train, y_test = train_test_split(x_tfidf, y, random_state=42, test_size=0.2)

c:\Users\Playdata\AppData\Local\miniconda3\envs\new_01\lib\site-packages\sklearn\feature_extraction\text.py:517: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(


In [11]:
from sklearn.naive_bayes import MultinomialNB
nb = MultinomialNB(alpha=0.1)
nb.fit(x_train, y_train)
nb.score(x_train, y_train), nb.score(x_test, y_test)

(1.0, 0.6666666666666666)

In [18]:
from sklearn.datasets import fetch_20newsgroups
# 20개토픽중에 선택 ( 무신론, 종교, 컴퓨터그래픽, 우주과학)
categories = ['alt.atheism','talk.religion.misc','sci.space']
news_train = fetch_20newsgroups(subset='train', remove=('headers','footers','quotes'), categories=categories)
news_test = fetch_20newsgroups(subset='test', remove=('headers','footers','quotes'), categories=categories)

In [19]:
len(news_train.data), len(news_test.data), set(news_test.target)

(1450, 964, {0, 1, 2})

In [20]:
x_train = news_train.data
y_train = news_train.target
x_test = news_test.data
y_test = news_test.target

In [21]:
# NPL문자 -> 학습 가능한 형태의 숫자 (vector) -> 모델학습
from sklearn.feature_extraction.text import CountVectorizer
cv = CountVectorizer(max_features=2000)
x_train_cv = cv.fit_transform(x_train)
x_test_cv = cv.transform(x_test)
x_test_cv.shape, x_train_cv.shape

((964, 2000), (1450, 2000))

In [22]:
nb.fit(x_train_cv, y_train)

,alpha,0.1
,force_alpha,True
,fit_prior,True
,class_prior,None


In [24]:
nb.score(x_train_cv, y_train), nb.score(x_test_cv, y_test)

(0.8317241379310345, 0.7365145228215768)

In [25]:
from sklearn.feature_extraction.text import TfidfVectorizer
tfidf = TfidfVectorizer(max_features=2000)
x_train_tfidf = tfidf.fit_transform(x_train)
x_test_tfidf = tfidf.transform(x_test)
x_train_tfidf.shape, x_test_tfidf.shape

((1450, 2000), (964, 2000))

In [26]:
nb.fit(x_train_tfidf, y_train)

,alpha,0.1
,force_alpha,True
,fit_prior,True
,class_prior,None


In [27]:
nb.score(x_train_tfidf, y_train), nb.score(x_test_tfidf, y_test)

(0.9110344827586206, 0.7396265560165975)

In [28]:
from sklearn.linear_model import LogisticRegression, RidgeClassifier, LassoCV
logistic = LogisticRegression()
ridge = RidgeClassifier()
lasso = LassoCV()

logistic.fit(x_train_tfidf, y_train)
ridge.fit(x_train_tfidf, y_train)
lasso.fit(x_train_tfidf, y_train)

print( logistic.score(x_train_tfidf, y_train), logistic.score(x_test_tfidf, y_test) )
print( ridge.score(x_train_tfidf, y_train), ridge.score(x_test_tfidf, y_test) )
print( lasso.score(x_train_tfidf, y_train), lasso.score(x_test_tfidf, y_test) )

0.926896551724138 0.7230290456431535
0.9634482758620689 0.7219917012448133
0.5078886999555219 0.10345219927883575


In [30]:
import numpy as np
alpha_lists = np.linspace(0.1, 10, 50)
params = {'alpha':alpha_lists}
from sklearn.model_selection import GridSearchCV
grid = GridSearchCV(RidgeClassifier(), param_grid=params)
grid.fit(x_train_tfidf, y_train)

,estimator,RidgeClassifier()
,param_grid,{'alpha': array([ 0.1 ... 10. ])}
,scoring,None
,n_jobs,None
,refit,True
,cv,None
,verbose,0
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,alpha,0.9081632653061225


In [31]:
best_model = grid.best_estimator_
best_model.score(x_train_tfidf,y_train), best_model.score(x_test_tfidf,y_test)

(0.9662068965517241, 0.7188796680497925)

In [ ]:
import sys
import os
sys.path.append(os.path.abspath("./"))

from nnst import downloader as downloader
import pprint
import argparse
import nnst.nnst as nnst

parser = argparse.ArgumentParser()

parser.add_argument('--csv_path', type=str)
parser.add_argument('--date', type=str)
parser.add_argument('--num', type=int)
parser.add_argument('--num_train', type=int)

# Jupyter 대응
args, unknown = parser.parse_known_args()

print(args)

csv_path = 'NNST_data.csv'
date = '20180914'
num = 1000
num_train = 900

print(parser.format_help())

# Namespace -> dict 변환
args = vars(args)

if args['csv_path'] is not None:
    csv_path = str(args['csv_path'])

if args['date'] is not None:
    date = str(args['date'])

if args['num'] is not None:
    num = int(args['num'])

if args['num_train'] is not None:
    num_train = int(args['num_train'])

downloader.download(num, csv_path, date)

data = nnst.load_data(csv_path)

train, test = nnst.div_dataset(data, train_size=num_train)

Namespace(csv_path=None, date=None, num=None, num_train=None)
usage: ipykernel_launcher.py [-h] [--csv_path CSV_PATH] [--date DATE]
                             [--num NUM] [--num_train NUM_TRAIN]

options:
  -h, --help            show this help message and exit
  --csv_path CSV_PATH
  --date DATE
  --num NUM
  --num_train NUM_TRAIN

